In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dev.bronze_db;

In [0]:
%sql
DROP TABLE IF EXISTS dev.bronze_db.customers;
DROP TABLE IF EXISTS dev.bronze_db.products;
DROP TABLE IF EXISTS dev.bronze_db.orders;
DROP TABLE IF EXISTS dev.silver_db.customers;
DROP TABLE IF EXISTS dev.silver_db.products;
DROP TABLE IF EXISTS dev.silver_db.orders;

In [0]:
%python
# Import necessary modules
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, StringType, DateType

# Define the storage path
customers_path = "/Volumes/datascience_catalog/default/day_0/Customers_*.parquet"
products_path = "/Volumes/datascience_catalog/default/day_0/Products_*.parquet"
orders_path = "/Volumes/datascience_catalog/default/day_0/Orders_*.parquet"
# Read the Parquet file without enforcing schema
df_customers = spark.read.parquet(customers_path)
df_products = spark.read.parquet(products_path)
df_orders = spark.read.parquet(orders_path)

# Print the schema to check actual column types
df_customers.printSchema()
df_products.printSchema()
df_orders.printSchema()

# For customers table convert Customer_ID to INT and start_date to DATE
df_customers = df_customers.withColumn("Customer_ID", col("Customer_ID").cast(IntegerType()))
df_customers = df_customers.withColumn("start_date", col("start_date").cast(DateType()))

# For products table convert start_date to DATE
df_products = df_products.withColumn("start_date", col("start_date").cast(DateType()))

# For orders table convert Customer_ID to INT and start_date to DATE
df_orders = df_orders.withColumn("Customer_ID", col("Customer_ID").cast(IntegerType()))
df_orders = df_orders.withColumn("start_date", col("start_date").cast(DateType()))

# Show DataFrame contents after type conversion
df_customers.show(truncate=False)
df_products.show(truncate=False)
df_orders.show(truncate=False)

# Ensure Delta table exists in Unity Catalog
spark.sql("""
CREATE TABLE IF NOT EXISTS dev.bronze_db.customers(
  Customer_ID INT,
  Customer_Name STRING,
  start_date DATE
) USING DELTA;
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS dev.bronze_db.products(
    Product_ID STRING,
    Product_Name STRING,
    start_date DATE
) USING DELTA;
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS dev.bronze_db.orders(
    Order_ID STRING,
    Customer_ID INT,
    Product_ID STRING,
    start_date DATE
) USING DELTA;
""")

# Write the transformed DataFrame to the Delta table
df_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev.bronze_db.customers")
df_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev.bronze_db.products")
df_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev.bronze_db.orders")


root
 |-- Customer_ID: long (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- start_date: timestamp_ntz (nullable = true)

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- start_date: timestamp_ntz (nullable = true)

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: long (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- start_date: timestamp_ntz (nullable = true)

+-----------+-------------+----------+
|Customer_ID|Customer_Name|start_date|
+-----------+-------------+----------+
|1          |Alice        |2025-03-29|
|2          |Bob          |2025-03-29|
|3          |Charlie      |2025-03-29|
|4          |David        |2025-03-29|
|5          |Ella         |2025-03-29|
|6          |Frank        |2025-03-29|
|7          |Grace        |2025-03-29|
|8          |Hannah       |2025-03-29|
|9          |Ian          |2025-03-29|
|10         |Jane         |2025-03-29|
+-----------+-------------+------

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dev.silver_db;

In [0]:
%sql
-- Create Silver Customers Table
CREATE TABLE IF NOT EXISTS dev.silver_db.customers (
  Customer_SK INT, -- Surrogate key (manually generated)
  Customer_ID INT NOT NULL,
  Customer_Name STRING NOT NULL,
  start_date TIMESTAMP,
  Load_date TIMESTAMP
) USING DELTA;

-- Create Silver Products Table
CREATE TABLE IF NOT EXISTS dev.silver_db.products (
  Product_SK INT, -- Surrogate key (manually generated)
  Product_ID STRING NOT NULL,
  Product_Name STRING NOT NULL,
  start_date TIMESTAMP,
  Load_date TIMESTAMP
) USING DELTA;

-- Create Silver Orders Table (Replacing IDs with Surrogate Keys)
CREATE TABLE IF NOT EXISTS dev.silver_db.orders (
  Order_ID STRING NOT NULL,
  Customer_SK INT NOT NULL,  -- Replaces Customer_ID
  Product_SK INT NOT NULL,   -- Replaces Product_ID
  start_date TIMESTAMP,
  Load_date TIMESTAMP,
  End_date TIMESTAMP  -- Renamed from Active_Until
) USING DELTA;


In [0]:
%sql
INSERT OVERWRITE dev.silver_db.customers
SELECT 
  1000 + ROW_NUMBER() OVER (ORDER BY Customer_ID) AS Customer_SK,
  Customer_ID,
  Customer_Name,
  start_date,
  DATE_FORMAT(CURRENT_TIMESTAMP, 'yyyy-MM-dd\'T\'HH:mm:ss') AS Load_date
FROM dev.bronze_db.customers
WHERE Customer_ID IS NOT NULL AND Customer_Name IS NOT NULL;


num_affected_rows,num_inserted_rows
10,10


In [0]:
%sql
INSERT OVERWRITE dev.silver_db.products
SELECT 
  2000 + ROW_NUMBER() OVER (ORDER BY Product_ID) AS Product_SK,
  Product_ID,
  Product_Name,
  start_date,
  DATE_FORMAT(CURRENT_TIMESTAMP, 'yyyy-MM-dd\'T\'HH:mm:ss') AS Load_date
FROM dev.bronze_db.products
WHERE Product_ID IS NOT NULL AND Product_Name IS NOT NULL;


num_affected_rows,num_inserted_rows
10,10


In [0]:
%sql
INSERT OVERWRITE dev.silver_db.orders
SELECT 
  o.Order_ID,
  c.Customer_SK,
  p.Product_SK,
  o.start_date,
  DATE_FORMAT(CURRENT_TIMESTAMP, 'yyyy-MM-dd\'T\'HH:mm:ss') AS Load_date,
  DATE_FORMAT('2999-12-31 23:59:59', 'yyyy-MM-dd\'T\'HH:mm:ss') AS End_date
FROM dev.bronze_db.orders o
JOIN dev.silver_db.customers c ON o.Customer_ID = c.Customer_ID
JOIN dev.silver_db.products p ON o.Product_ID = p.Product_ID
WHERE o.Order_ID IS NOT NULL AND o.Customer_ID IS NOT NULL AND o.Product_ID IS NOT NULL;

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
select * from dev.bronze_db.orders

Order_ID,Customer_ID,Product_ID,start_date
O101,1,P101,2025-03-29
O102,2,P102,2025-03-29
O103,3,P103,2025-03-29
O104,4,P104,2025-03-29
O105,5,P105,2025-03-29


In [0]:
%sql
select * from dev.silver_db.orders

Order_ID,Customer_SK,Product_SK,start_date,Load_date,End_date
O101,1001,2001,2025-03-29T00:00:00.000Z,2025-03-30T18:37:47.000Z,2999-12-31T23:59:59.000Z
O102,1002,2002,2025-03-29T00:00:00.000Z,2025-03-30T18:37:47.000Z,2999-12-31T23:59:59.000Z
O103,1003,2003,2025-03-29T00:00:00.000Z,2025-03-30T18:37:47.000Z,2999-12-31T23:59:59.000Z
O104,1004,2004,2025-03-29T00:00:00.000Z,2025-03-30T18:37:47.000Z,2999-12-31T23:59:59.000Z
O105,1005,2005,2025-03-29T00:00:00.000Z,2025-03-30T18:37:47.000Z,2999-12-31T23:59:59.000Z
